In [24]:
import pandas as pd
import numpy as np
import re

df = pd.read_excel('ai.xlsx')

df['MRP'] = df['MRP'].str.replace('₹', '').str.replace(' ', '').str.replace(',', '').astype(float)
df['Current Price'] = df['Current Price'].str.replace('₹', '').str.replace(',', '').astype(float)

def extract_discount(value):
    if pd.isna(value):
        return 0  # or return None
    return int(value.replace('% off', ''))
df['Discount_Number'] = df['Discount'].apply(extract_discount)

df.drop("Discount", axis=1, inplace=True)

brands = {
    'Apple': ['apple', 'iphone', 'ipad'],
    'motorola': ['motorola', 'moto'],
    'AI_plus': ['ai+', 'ai', 'aiplus', 'ai plus', 'a i'], 
    'IQOO': ['iqoo'],
    'Tecno': ['tecno'],
    'lava': ['lava'],
    'readmi': ['readmi', 'redmi'],  
    'realme': ['realme'],
    'POCO': ['poco'],
    'nothing': ['nothing'],
    'Infix': ['infix'],
    'Oppo': ['oppo'],
    'vivo': ['vivo'],
    'Google': ['google', 'pixel'],
    'OnePlus': ['oneplus', 'one plus'],
    'Samsung': ['samsung', 'galaxy'],
}

def get_brand(product_name):
    if pd.isna(product_name):
        return 'Unknown'
    
    product_name = str(product_name).lower()
    
    for brand, keywords in brands.items():
        for keyword in keywords:
            if keyword in product_name:
                return brand
    return 'Unknown'
df['Brand'] = df['Product Name'].apply(get_brand)

df['RAM'] = df['RAM'].str.extract(r'(\d+)').astype(int)


df['Storage'] = df['Storage'].str.extract(r'(\d+)').astype(int)


def extract_ratings_reviews(text):
    """
    Extract ratings and reviews from text
    Example: "2,46,375 Ratings & 9,265 Reviews"
    Returns: (ratings, reviews)
    """
    if pd.isna(text):
        return None, None
    
    text = str(text)
    
    # Extract ratings (numbers before 'Ratings')
    ratings_match = re.search(r'([\d,]+)\s*Ratings', text, re.IGNORECASE)
    ratings = ratings_match.group(1).replace(',', '') if ratings_match else None
    
    # Extract reviews (numbers before 'Reviews')
    reviews_match = re.search(r'([\d,]+)\s*Reviews', text, re.IGNORECASE)
    reviews = reviews_match.group(1).replace(',', '') if reviews_match else None
    
    return ratings, reviews

# Apply to dataframe
df[['Ratings', 'Reviews']] = df['Review text'].apply(
    lambda x: pd.Series(extract_ratings_reviews(x))
)

# Convert to numbers
df['Ratings'] = pd.to_numeric(df['Ratings'], errors='coerce')
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')


def extract_battery(text):
    if pd.isna(text):
        return 0
    match = re.search(r'(\d+)', str(text))
    return int(match.group()) if match else 0

df['Battery'] = df['Battery'].apply(extract_battery)

In [25]:
df.head()

,Product Name,Current Price,MRP,Rating,Review text,RAM,Storage,Display,Camera,Battery,Processor,Product URL,Image URL,Discount_Number,Brand,Ratings,Reviews
0,"Ai+ Nova 2 Ultra 5G (Purple, 128 GB)",17999.0,25999.0,4.1,903 Ratings & 88 Reviews,6,128,17.22 cm (6.78 inch) Full HD+ AMOLED Display,50MP + 8MP | 13MP Front Camera,6000,Dimensity 7400 Processor,https://www.flipkart.com/ai-nova-2-ultra-5g-pu...,https://rukminim2.flixcart.com/image/312/312/x...,30,AI_plus,903.0,88.0
1,"Ai+ Pulse 1 (Blue, 64 GB)",7999.0,NaN,4.2,"43,123 Ratings & 2,831 Reviews",4,64,17.13 cm (6.745 inch) HD+ Display,50MP Rear Camera | 5MP Front Camera,5000,T615 Processor,https://www.flipkart.com/ai-pulse-1-blue-64-gb...,https://rukminim2.flixcart.com/image/312/312/x...,0,AI_plus,43123.0,2831.0
2,"Ai+ Nova 2 Ultra 5G (Red, 128 GB)",19999.0,26999.0,4.0,413 Ratings & 53 Reviews,8,128,17.22 cm (6.78 inch) Full HD+ AMOLED Display,50MP + 8MP | 13MP Front Camera,6000,Dimensity 7400 Processor,https://www.flipkart.com/ai-nova-2-ultra-5g-re...,https://rukminim2.flixcart.com/image/312/312/x...,25,AI_plus,413.0,53.0
3,"Ai+ Pulse 1 (Black, 64 GB)",7999.0,NaN,4.2,"43,123 Ratings & 2,831 Reviews",4,64,17.13 cm (6.745 inch) HD+ Display,50MP Rear Camera | 5MP Front Camera,5000,T615 Processor,https://www.flipkart.com/ai-pulse-1-black-64-g...,https://rukminim2.flixcart.com/image/312/312/x...,0,AI_plus,43123.0,2831.0
4,"Ai+ Nova 2 Ultra 5G (Green, 128 GB)",19999.0,26999.0,4.0,413 Ratings & 53 Reviews,8,128,17.22 cm (6.78 inch) Full HD+ AMOLED Display,50MP + 8MP | 13MP Front Camera,6000,Dimensity 7400 Processor,https://www.flipkart.com/ai-nova-2-ultra-5g-gr...,https://rukminim2.flixcart.com/image/312/312/x...,25,AI_plus,413.0,53.0


In [26]:
df.isnull().sum()

Product Name        0
Current Price       0
MRP                34
Rating             20
Review text        20
RAM                 0
Storage             0
Display             0
Camera              0
Battery             0
Processor           1
Product URL         0
Image URL           0
Discount_Number     0
Brand               0
Ratings            20
Reviews            20
dtype: int64

In [27]:
df['MRP'] = np.where(
    (df['Discount_Number'] == 0) & (df['MRP'].isna() | (df['MRP'] == '')),
    df['Current Price'],
    df['MRP']
)

In [28]:
df.isnull().sum()

Product Name        0
Current Price       0
MRP                 0
Rating             20
Review text        20
RAM                 0
Storage             0
Display             0
Camera              0
Battery             0
Processor           1
Product URL         0
Image URL           0
Discount_Number     0
Brand               0
Ratings            20
Reviews            20
dtype: int64

In [29]:
df['Rating'] = df['Rating'].fillna(df['Rating'].median())
df['Ratings'] = df['Ratings'].fillna(df['Ratings'].median()).astype(int)
df['Reviews'] = df['Reviews'].fillna(df['Reviews'].median()).astype(int)

In [30]:
df.isnull().sum()

Product Name        0
Current Price       0
MRP                 0
Rating              0
Review text        20
RAM                 0
Storage             0
Display             0
Camera              0
Battery             0
Processor           1
Product URL         0
Image URL           0
Discount_Number     0
Brand               0
Ratings             0
Reviews             0
dtype: int64

In [32]:
df.to_excel('Cleaned_ai.xlsx', index=False)